In [3]:
# Cell 1: Imports and Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import pyedflib
import os
import warnings
warnings.filterwarnings('ignore')

# Machine learning imports
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, LeaveOneGroupOut
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler

# Set random seed for reproducibility
np.random.seed(1)



In [31]:
# Cell 2: Define seizures, channel names, and helper functions

# All 7 seizures for chb01
seizures_all = [
    {'file': 'chb01_03.edf', 'start': 2996, 'end': 3036},
    {'file': 'chb01_04.edf', 'start': 1467, 'end': 1494},
    {'file': 'chb01_15.edf', 'start': 1732, 'end': 1772},
    {'file': 'chb01_16.edf', 'start': 1015, 'end': 1066},
    {'file': 'chb01_18.edf', 'start': 1720, 'end': 1810},
    {'file': 'chb01_21.edf', 'start': 327,  'end': 420},
    {'file': 'chb01_26.edf', 'start': 1862, 'end': 1963},
]

# Clean baseline files (no seizures, at least 1 hour from any seizure)
clean_files = [
    'chb01_07.edf',
    'chb01_08.edf',
    'chb01_09.edf',
    'chb01_10.edf',
    'chb01_11.edf',
    'chb01_12.edf',
    'chb01_13.edf',
]

# Channel names from CHB-MIT summary file
channel_names = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ',  'CZ-PZ', 'P7-T7', 'T7-FT9',
    'FT9-FT10', 'FT10-T8', 'T8-P8'
]

# Setup
data_dir   = '../data/chb01/'
sfreq      = 256
n_channels = 23

print(f"  Seizures: {len(seizures_all)}")
print(f"  Clean files: {len(clean_files)}")
print(f"  Channels: {n_channels}")
print(f"  Features per channel: 3 (entropy, line length, high-freq power)")
print(f"  Total features per window: {n_channels * 3}")

  Seizures: 7
  Clean files: 7
  Channels: 23
  Features per channel: 3 (entropy, line length, high-freq power)
  Total features per window: 69


In [21]:
# Cell 3: Feature Extraction Functions

def extract_eeg_from_file(filename, channel_idx):
    filepath = os.path.join(data_dir, filename)
    f = pyedflib.EdfReader(filepath)
    eeg_signal = f.readSignal(channel_idx)
    f.close()
    return eeg_signal

def calculate_entropy(signal_segment):
    hist, _ = np.histogram(signal_segment, bins=50)
    hist = hist / np.sum(hist)
    hist = hist[hist > 0]
    entropy_val = -np.sum(hist * np.log2(hist))
    return entropy_val

def calculate_line_length(signal_segment):
    differences = np.diff(signal_segment)
    absolute_differences = np.abs(differences)
    line_length_val = np.sum(absolute_differences)
    return line_length_val

def calculate_high_freq_power(signal_segment, sfreq=256):
    nperseg = min(1024, len(signal_segment))
    f, pxx = signal.welch(signal_segment, sfreq, nperseg=nperseg)
    high_freq_power = np.sum(pxx[f >= 20])
    return high_freq_power

# Returns a 69-element array (3 features x 23 channels) wich is one row of X
def extract_features_from_window(eeg_all_channels, start_sec, end_sec, sfreq):
    """
    Extract all 69 features from a 2-minute window across all 23 channels.
    
    Parameters:
    - eeg_all_channels: list of 23 EEG signals (one per channel)
    - start_sec: window start time in seconds
    - end_sec: window end time in seconds
    - sfreq: sampling frequency (256 Hz)
    
    Returns:
    - feature_vector: numpy array of 69 numbers
    - feature_names: list of 69 feature names
    """
    feature_vector = [] # will hold 69 numbers
    feature_names  = [] # will hold 69 names like FP1-F7_entropy

    # loop through the 23 channels
    for ch_idx, ch_name in enumerate(channel_names):
        if eeg_all_channels[ch_idx] is None:
            # If channel failed to load fill with zeros
            feature_vector.extend([0, 0, 0])
            feature_names.extend([
                f'{ch_name}_entropy',
                f'{ch_name}_line_length',
                f'{ch_name}_hf_power'
            ])
            continue
        
        # Convert time in seconds to sample index (256 samples per second)
        start_idx = int(start_sec * sfreq)
        end_idx   = int(end_sec   * sfreq)

        # slice out the 2-min segment for this channel
        seg = eeg_all_channels[ch_idx][start_idx:end_idx]
        
        if len(seg) == 0:
            feature_vector.extend([0, 0, 0])
        else:
            # extract all 3 features from this 2-minute segment
            e  = calculate_entropy(seg)
            ll = calculate_line_length(seg)
            hf = calculate_high_freq_power(seg, sfreq)
            # add to the feature vector
            feature_vector.extend([e, ll, hf])

        # store the feature names for later use
        feature_names.extend([
            f'{ch_name}_entropy',
            f'{ch_name}_line_length',
            f'{ch_name}_hf_power'
        ])

    # return an array of 69 features (23 channels * 3 features)
    return np.array(feature_vector), feature_names

In [26]:
# Cell 4: Build Labeled Dataset for chb01 (all 7 seizures with stitching)

import random
random.seed(1)

WINDOW_SEC   = 120   # 2-minute windows
STEP_SEC     = 120   # non-overlapping
PREICTAL_SEC = 1200  # 20 minutes before seizure = preictal zone
WINDOWS_PER_FILE = 10 # 10 interictal windows per clean file

# map each seizure to its previous file for stitching
seizure_prev_files = {
    0: None,            # seizure 1 has enough data (2996s > 1200s)
    1: 'chb01_03.edf',  # seizure 2 needs chb01_03
    2: 'chb01_14.edf',  # seizure 3 needs chb01_14
    3: 'chb01_15.edf',  # seizure 4 needs chb01_15
    4: 'chb01_17.edf',  # seizure 5 needs chb01_17
    5: 'chb01_20.edf',  # seizure 6 needs chb01_20
    6: 'chb01_25.edf',  # seizure 7 needs chb01_25
}

X = []
y = []
feature_names_list = None

print("Building labeled dataset for chb01 (all 7 seizures with stitching)...\n")

# ─── PREICTAL WINDOWS ───────────────────────────────────────────
print("Extracting PREICTAL windows...")
for sz_idx, seizure in enumerate(seizures_all):
    sz_start  = seizure['start']
    sz_file   = seizure['file']
    prev_file = seizure_prev_files[sz_idx]
    
    preictal_start = sz_start - PREICTAL_SEC  # 20 min before onset
    preictal_end   = sz_start
    
    print(f"\n  Seizure {sz_idx+1} ({sz_file}, onset {sz_start}s)")
    
    # load all 23 channels — stitch if needed
    eeg_channels = []
    
    for ch in range(n_channels):
        try:
            # load current seizure file
            eeg_current = extract_eeg_from_file(sz_file, ch)
            
            if preictal_start < 0 and prev_file is not None:
                # preictal zone starts before current file
                # need to stitch previous file
                need_from_prev = abs(preictal_start)
                eeg_prev = extract_eeg_from_file(prev_file, ch)
                
                # take last need_from_prev seconds from previous file
                prev_samples = int(need_from_prev * sfreq)
                eeg_prev_section = eeg_prev[-prev_samples:]
                
                # take from start of current file up to seizure onset
                eeg_current_section = eeg_current[:int(sz_start * sfreq)]
                
                # stitch together
                eeg_stitched = np.concatenate([eeg_prev_section, eeg_current_section])
                eeg_channels.append(eeg_stitched)
                
            else:
                # enough data in current file — no stitching needed
                eeg_channels.append(eeg_current)
                
        except Exception as e:
            eeg_channels.append(None)
    
    # now extract preictal windows
    # if stitched: preictal zone starts at 0 in the stitched signal
    # if not stitched: preictal zone starts at preictal_start in current file
    if preictal_start < 0 and prev_file is not None:
        # stitched signal — preictal zone is the full stitched signal
        extract_start = 0
        extract_end   = PREICTAL_SEC
    else:
        # no stitching — use original timestamps
        extract_start = preictal_start
        extract_end   = preictal_end
    
    current_time = extract_start
    n_windows = 0
    
    while current_time + WINDOW_SEC <= extract_end:
        fv, fn = extract_features_from_window(
            eeg_channels, current_time, current_time + WINDOW_SEC, sfreq
        )
        X.append(fv)
        y.append(1)
        
        if feature_names_list is None:
            feature_names_list = fn
        
        current_time += STEP_SEC
        n_windows += 1
    
    print(f"    → {n_windows} preictal windows")

# ─── INTERICTAL WINDOWS ─────────────────────────────────────────
print(f"\nExtracting INTERICTAL windows ({WINDOWS_PER_FILE} random per file)...")
for clean_file in clean_files:
    print(f"  {clean_file}...")
    
    eeg_channels = []
    for ch in range(n_channels):
        try:
            eeg_channels.append(extract_eeg_from_file(clean_file, ch))
        except:
            eeg_channels.append(None)
    
    max_start = 3600 - WINDOW_SEC
    possible_starts = list(range(0, max_start, WINDOW_SEC))
    random_starts = random.sample(possible_starts, WINDOWS_PER_FILE)
    random_starts.sort()
    
    n_windows = 0
    for start in random_starts:
        fv, fn = extract_features_from_window(
            eeg_channels, start, start + WINDOW_SEC, sfreq
        )
        X.append(fv)
        y.append(0)
        n_windows += 1
    
    print(f"    → {n_windows} interictal windows")

# Convert to numpy arrays
X = np.array(X)
y = np.array(y)

print(f"\n✓ Dataset built")
print(f"  Total windows:      {len(y)}")
print(f"  Preictal (y=1):     {np.sum(y == 1)}")
print(f"  Interictal (y=0):   {np.sum(y == 0)}")
print(f"  X shape:            {X.shape}")
print(f"  y shape:            {y.shape}")
print(f"  Class ratio:        {np.sum(y==1)/len(y):.2f} preictal")

Building labeled dataset for chb01 (all 7 seizures with stitching)...

Extracting PREICTAL windows...

  Seizure 1 (chb01_03.edf, onset 2996s)
    → 10 preictal windows

  Seizure 2 (chb01_04.edf, onset 1467s)
    → 10 preictal windows

  Seizure 3 (chb01_15.edf, onset 1732s)
    → 10 preictal windows

  Seizure 4 (chb01_16.edf, onset 1015s)
    → 10 preictal windows

  Seizure 5 (chb01_18.edf, onset 1720s)
    → 10 preictal windows

  Seizure 6 (chb01_21.edf, onset 327s)
    → 10 preictal windows

  Seizure 7 (chb01_26.edf, onset 1862s)
    → 10 preictal windows

Extracting INTERICTAL windows (10 random per file)...
  chb01_07.edf...
    → 10 interictal windows
  chb01_08.edf...
    → 10 interictal windows
  chb01_09.edf...
    → 10 interictal windows
  chb01_10.edf...
    → 10 interictal windows
  chb01_11.edf...
    → 10 interictal windows
  chb01_12.edf...
    → 10 interictal windows
  chb01_13.edf...
    → 10 interictal windows

✓ Dataset built
  Total windows:      140
  Preictal

In [30]:
# Cell 5: Train Random Forest Classifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler

# SCALE FEATURES
# Standard scaler converts every feature 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"  X shape: {X_scaled.shape}")
print(f"  y shape: {y.shape}")

# Define groups 
# Groups tell the cross validator which windows belong to which seizure
# Leaving out seizure 1, all windows in group 0 are not used
# Dynamically build groups based on actual dataset

# how many 2-min windows fit in preictal zone
# Ex. 1200 sec (20 min) / 120 seconds  = 10 windows per seizure
windows_per_seizure = int(PREICTAL_SEC / WINDOW_SEC)  # 10

groups = []

# Preictal windows — one group per seizure
# ALL 7 seizures now valid since we stitch previous files
for sz_idx in range(len(seizures_all)):
    for _ in range(windows_per_seizure):
        groups.append(sz_idx)

        
# Interictal windows: split evenly across seizure groups
# Interictal windows — round robin across all 7 seizure groups
n_interictal = len(y) - len(groups)
valid_seizure_indices = list(range(len(seizures_all)))  # [0,1,2,3,4,5,6]
n_valid_seizures = len(valid_seizure_indices)

# Assign interictal windows round-robin across seizure groups
# example:
# interictal window 1  -> seizure group 0 (seizure 1)
# interictal window 2  -> seizure group 1 (seizure 2)
# interictal window 3  -> seizure group 2 (seizure 3)
# interictal window 4  -> seizure group 4 (seizure 5)
# interictal window 5  -> seizure group 6 (seizure 7)
# interictal window 6  -> seizure group 0 (back to seizure 1)
# repeating for all 70 seizures -> will make it so that each group has 14 interictal seizures
for i in range(n_interictal):
    groups.append(valid_seizure_indices[i % n_valid_seizures])

groups = np.array(groups)

print(f"  Groups array length: {len(groups)}")
print(f"  Matches X and y: {len(groups) == len(X) == len(y)}")
print(f"  Unique groups: {np.unique(groups)}")
print(f"  Windows per seizure: {windows_per_seizure}")

# TRAIN RANDOM FOREST
rf = RandomForestClassifier(
    n_estimators=100, # builds 100 decision trees
    class_weight='balanced', # weight preictal mistakes more heavily
    random_state=1, # fixed state to get reproducible results
    n_jobs=-1
)

# leave one seizure out at a time
# model retrained from scratch each time
logo = LeaveOneGroupOut()

# cross validation
# runs all 5 rounds
# testing the model multiple times based on the data
y_pred = cross_val_predict(
    rf, X_scaled, y,
    groups=groups,
    cv=logo
)


# Results
print("RANDOM FOREST RESULTS for chb01")

print("\nClassification Report:")
print(classification_report(y, y_pred, target_names=['Interictal', 'Preictal']))

# confusion matrix shows what the model got right and wrong
cm = confusion_matrix(y, y_pred)
print("Confusion Matrix:")
print(f"                   Predicted")
print(f"                   Interictal  Preictal")
print(f"Actual Interictal      {cm[0][0]}          {cm[0][1]}")
print(f"Actual Preictal        {cm[1][0]}          {cm[1][1]}")

# TP = true positive
# TN = true negative
# Sensitivity =  TP / (TP + FN)
# correctly predicted preictal / all preictal windows
sensitivity = cm[1][1] / (cm[1][1] + cm[1][0])

# Specificity = TN / (TN + FP)
# = correctly predicted interictal / all actual interictal windows
specificity = cm[0][0] / (cm[0][0] + cm[0][1])

print(f"\nSensitivity: {sensitivity:.2%}")
print(f"Specificity: {specificity:.2%}")

# AUC measures overall ability to separate preictal from interictal
# 1.0 = perfect, 0.5 = random guessing
try:
    auc = roc_auc_score(y, y_pred)
    print(f"AUC:         {auc:.3f}")
except:
    print("does not work for some reason, auc is broken")

  X shape: (140, 69)
  y shape: (140,)
  Groups array length: 140
  Matches X and y: True
  Unique groups: [0 1 2 3 4 5 6]
  Windows per seizure: 10
RANDOM FOREST RESULTS for chb01

Classification Report:
              precision    recall  f1-score   support

  Interictal       0.95      0.99      0.97        70
    Preictal       0.99      0.94      0.96        70

    accuracy                           0.96       140
   macro avg       0.97      0.96      0.96       140
weighted avg       0.97      0.96      0.96       140

Confusion Matrix:
                   Predicted
                   Interictal  Preictal
Actual Interictal      69          1
Actual Preictal        4          66

Sensitivity: 94.29%
Specificity: 98.57%
AUC:         0.964


In [28]:
print("Group distribution:")
for g in np.unique(groups):
    count = np.sum(groups == g)
    print(f"  Group {g}: {count} windows")
    

Group distribution:
  Group 0: 28 windows
  Group 1: 28 windows
  Group 2: 28 windows
  Group 4: 28 windows
  Group 6: 28 windows
